In [7]:

import re
import os
import json
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import random
import torch
import numpy as np

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

SEED=42
 
set_seed(SEED)

DATASET = "IMDB"
REMOVE_PUNCTUATION = False   # elimina , si .
REMOVE_STOPWORDS = True    # elimina stop words


In [8]:

from spacy.lang.en.stop_words import STOP_WORDS


def fix_spaced_contractions(text: str) -> str:
    text = re.sub(r"\b(\w+)\s+'\s*s\b", r"\1's", text)
    text = re.sub(r"\b(\w+)\s+'\s*re\b", r"\1're", text)
    text = re.sub(r"\b(\w+)\s+'\s*ve\b", r"\1've", text)
    text = re.sub(r"\b(\w+)\s+'\s*ll\b", r"\1'll", text)
    text = re.sub(r"\b(\w+)\s+'\s*d\b", r"\1'd", text)
    text = re.sub(r"\b(\w+)\s+'\s*m\b", r"\1'm", text)
    text = re.sub(r"\b(\w+)\s+n\s*'\s*t\b", r"\1n't", text)

    text = re.sub(r"\bca\s+n\s*'\s*t\b", "can't", text)
    text = re.sub(r"\bwo\s+n\s*'\s*t\b", "won't", text)
    text = re.sub(r"\bsha\s+n\s*'\s*t\b", "shan't", text)

    text = re.sub(r"\b(\w+)\s+'\s*em\b", r"\1 'em", text)

    return text


CONTRACTION_MAP = {
    "'em": "them",
    "'ve": " have",
    "aren't": "are not",
    "can't": "can not",
    "couldn't": "could not",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'll": "he will",
    "he's": "he is",
    "i'd": "i would",
    "i'll": "i will",
    "i'm": "i am",
    "i've": "i have",
    "isn't": "is not",
    "it's": "it is",
    "let's": "let us",
    "mightn't": "might not",
    "mustn't": "must not",
    "shan't": "shall not",
    "she'd": "she would",
    "she'll": "she will",
    "she's": "she is",
    "shouldn't": "should not",
    "that's": "that is",
    "there's": "there is",
    "they'd": "they would",
    "they'll": "they will",
    "they're": "they are",
    "they've": "they have",
    "wasn't": "was not",
    "we'd": "we would",
    "we'll": "we will",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what's": "what is",
    "where's": "where is",
    "who's": "who is",
    "won't": "will not",
    "wouldn't": "would not",
    "you'd": "you would",
    "you'll": "you will",
    "you're": "you are",
    "you've": "you have",
}

_contraction_pattern = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in sorted(CONTRACTION_MAP, key=len, reverse=True)) + r")\b"
)


def expand_contractions(text: str) -> str:
    return _contraction_pattern.sub(lambda m: CONTRACTION_MAP[m.group(0)], text)


def preprocess(text: str) -> str:
    text = text.lower()
    text = re.sub(r"<br\s*/?\s*>", " ", text)
    text = text.replace('"', ' ')
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = fix_spaced_contractions(text)
    text = expand_contractions(text)
    text = re.sub(r"\b(\w+)'s\b", r"\1", text)
    text = re.sub(r"\.{2,}|--|/|;|[()]", " ", text)

    if REMOVE_PUNCTUATION:
        text = text.replace(",", " ").replace(".", " ")

    text = re.sub(r"\s+", " ", text).strip()

    if REMOVE_STOPWORDS:
        text = " ".join(w for w in text.split() if w not in STOP_WORDS)

    return text


In [9]:

from collections import Counter

suffix_parts = []
if REMOVE_PUNCTUATION:
    suffix_parts.append("rm-punct")
if REMOVE_STOPWORDS:
    suffix_parts.append("rm-stop")
OUTPUT_DIR = DATASET if not suffix_parts else f"{DATASET}_{'_'.join(suffix_parts)}"

print(f"Output dir: {OUTPUT_DIR}")

if DATASET == "SST-2":
    dataset = load_dataset("glue", "sst2")
    TEXT_ORIG = "sentence_original"
    TEXT_PREP = "sentence_preprocessed"
    SRC_TEXT_COL = "sentence"

    splits_data = {}
    for split_name in ["train", "validation", "test"]:
        texts = dataset[split_name][SRC_TEXT_COL]
        labels = dataset[split_name]["label"]
        splits_data[split_name] = (texts, labels)
        print(f"  {split_name}: {len(texts)} samples")

else:
    dataset = load_dataset(DATASET.lower())
    TEXT_ORIG = "text_original"
    TEXT_PREP = "text_preprocessed"
    VAL_SIZE = 0.2

    # Detect source text column (fresh download = "text", cached/preprocessed = "text_original")
    cols = dataset["train"].column_names
    SRC_TEXT_COL = "text" if "text" in cols else "text_original"
    print(f"  Source text column: {SRC_TEXT_COL}")

    train_texts = dataset["train"][SRC_TEXT_COL]
    train_labels = dataset["train"]["label"]

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        train_texts, train_labels,
        test_size=VAL_SIZE,
        random_state=SEED,
        stratify=train_labels
    )

    splits_data = {
        "train": (train_texts, train_labels),
        "validation": (val_texts, val_labels),
        "test": (dataset["test"][SRC_TEXT_COL], dataset["test"]["label"]),
    }

    for split_name, (texts, labels) in splits_data.items():
        print(f"  {split_name}: {len(texts)} samples")
    print(f"\n  Train labels:      {dict(Counter(train_labels))}")
    print(f"  Validation labels: {dict(Counter(val_labels))}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

for split_name, (texts, labels) in splits_data.items():
    records = [
        {
            TEXT_ORIG: text,
            TEXT_PREP: preprocess(text),
            "label": label,
        }
        for text, label in zip(texts, labels)
    ]

    path = os.path.join(OUTPUT_DIR, f"{split_name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    print(f"{split_name:>10}: {len(records):>6} samples -> {path}")

print(f"\nDone!")


Output dir: IMDB_rm-stop
  Source text column: text_original
  train: 16000 samples
  validation: 4000 samples
  test: 25000 samples

  Train labels:      {1: 8000, 0: 8000}
  Validation labels: {1: 2000, 0: 2000}
     train:  16000 samples -> IMDB_rm-stop\train.json
validation:   4000 samples -> IMDB_rm-stop\validation.json
      test:  25000 samples -> IMDB_rm-stop\test.json

Done!
